# Основы алгоритмов: линейные приёмы

Этот ноутбук рассчитан на самостоятельное изучение. Сначала мы поймём, почему программы могут быть медленными, затем разберём четыре главных приёма: один проход по массиву, префиксные суммы, два указателя и скользящее окно. После теории идут 30 задач: 15 базовых, 10 средних и 5 сложных.

## Как устроена практика

Каждая задача оформлена одинаково: сначала отдельная ячейка с объяснением приёма, условием, форматом ввода и вывода, примерами и раскрывающимися подсказками. Сразу под ней находится функция taskN и её автотест.

Функция читает данные через input и печатает ответ через print. В шаблоне замените три точки на своё решение и запустите ячейку. Тесты лежат в файле tests_algorithms.py рядом с ноутбуком.

В примерах:

    -> строка входа
    --
    <- ожидаемый вывод


# Часть I. Сложность алгоритмов

Алгоритм — точный набор шагов для решения задачи. Для одной и той же задачи можно написать несколько программ: обе дадут верный ответ, но одна обработает сто тысяч чисел быстро, а другая будет работать слишком долго.

Размер входа обычно обозначают n. Запись O(...) описывает, как растёт число действий:

| Сложность | Смысл | Пример |
|---|---|---|
| O(1) | число действий не зависит от n | взять a[i] |
| O(n) | один проход по данным | найти максимум |
| O(n log n) | почти линейно | сортировка |
| O(n²) | проверка всех пар | два вложенных цикла |

При n около 100000 обычно годятся O(n) и O(n log n), но не O(n²). Важно считать не число строчек кода, а сколько раз реально выполняется действие.

Два вложенных цикла не всегда дают квадратную сложность. Если left и right только движутся вправо и никогда не возвращаются, каждый указатель пройдёт максимум n позиций. Вместе это O(n), именно так работают два указателя и скользящее окно.


# Часть II. Линейные алгоритмы

## 1. Сканирующая прямая

Сканирование — это проход по данным слева направо. В переменных сохраняют только информацию, нужную для следующего шага: текущий максимум, сумму, предыдущий элемент, длину текущей серии или лучший ответ.

### Рекорд

Для максимума переменная best после каждого шага хранит лучший элемент среди уже просмотренных. Если нужен первый максимум, обновляйте его только при строгом сравнении x > best; сравнение x >= best выберет последний максимум.

### Соседи и серии

В задачах со словами «рядом», «подряд», «серия» нужно помнить предыдущий элемент. Для серии одинаковых чисел current хранит длину серии, которая заканчивается в текущей позиции, а best хранит рекорд. При смене значения current становится 1, потому что текущий элемент уже образует серию.

### Состояние из двух значений

Для максимальной суммы подотрезка хранится лучшая сумма, оканчивающаяся здесь, и глобальный максимум. Для максимального произведения нужны текущий максимум и текущий минимум: отрицательное число может превратить самый маленький отрицательный результат в большой положительный.


In [ ]:
a = [3, -1, 7, 7, 0]
best, position = a[0], 0
for i, x in enumerate(a[1:], 1):
    if x > best:
        best, position = x, i
print(best, position)  # 7 2

## 2. Префиксные суммы

Пусть p[i] — сумма первых i элементов, причём p[0] = 0. Тогда сумма элементов с индексами от l до r включительно равна p[r + 1] - p[l]. Нулевой префикс особенно важен: он позволяет той же формулой считать отрезки, начинающиеся с нулевого индекса.

Префиксы полезны, когда есть много вопросов о суммах на отрезках: подготовка занимает O(n), а каждый запрос после этого — O(1).

Для подсчёта подотрезков с суммой S используйте словарь частот префиксов. Если текущая сумма cur, нужны старые префиксы cur - S. Сначала добавьте их количество к ответу, потом увеличьте частоту cur. Этот метод работает и с нулями, и с отрицательными числами.


### Примеры подслучаев

Для массива [2, -1, 3, 0, 4] префиксы равны [0, 2, 1, 4, 4, 8]. Сумма на отрезке с индексами 1..3: 4 - 2 = 2.

Равновесный индекс в [1, 3, 5, 2, 2] равен 2: слева 1 + 3 = 4, справа 2 + 2 = 4.

Для [1, 2, 1, 2, 1] и суммы 3 подходят четыре отрезка: [1, 2], [2, 1], [1, 2], [2, 1]. Одинаковые префиксные суммы нельзя хранить просто во множестве: нужно помнить их количество.


In [ ]:
a = [5, -2, 4, 1]
p = [0]
for x in a:
    p.append(p[-1] + x)
print(p)
print(p[3] - p[1])  # сумма от -2 до 4

## 3. Два указателя

В отсортированном массиве для пары с суммой X поставьте left в начало, right в конец. Если сумма мала, двигайте left вправо; если велика — right влево. Причина: после этого шага отброшенная пара уже точно не сможет стать ответом.

Тот же приём сливает два отсортированных списка: берём меньший из двух текущих элементов. Для ближайшей пары из двух списков двигаем указатель того значения, которое сейчас меньше. Для палиндрома указатели идут с краёв строки к центру.

Всегда проверяйте, что указатели не пересеклись. Для пары нужны два разных индекса, поэтому условие цикла left < right.


### Примеры подслучаев

Пара с суммой: в [1, 3, 4, 7, 11] для цели 10 сначала 1 + 11 слишком много, поэтому правый указатель идёт к 7; затем 1 + 7 мало, левый идёт к 3; 3 + 7 даёт ответ.

Слияние: из [1, 4, 8] и [2, 3, 7, 9] последовательно выбираются 1, 2, 3, 4, 7, 8, затем дописывается хвост 9.

Палиндром: строку «А роза упала на лапу Азора» сначала приводят к нижнему регистру и убирают пробелы, затем сравнивают одинаковые символы с краёв.


In [ ]:
a, target = [1, 3, 4, 7, 11], 10
left, right = 0, len(a) - 1
while left < right:
    total = a[left] + a[right]
    if total == target:
        print(a[left], a[right])
        break
    if total < target:
        left += 1
    else:
        right -= 1

## 4. Скользящее окно

Окно — непрерывный фрагмент от left до right. Правый указатель добавляет элемент, левый убирает лишнее. Так находят самый длинный кусок с суммой не больше K, минимальный кусок с суммой не меньше S, подстроку без повторов и отрезки с ограниченным числом разных значений.

Для самого длинного допустимого окна сначала расширяем right, затем пока условие нарушено сдвигаем left, а потом обновляем максимум длины. Для самого короткого достаточного окна всё наоборот: как только условие выполнено, стараемся сжать окно.

Приём с суммой применим только к неотрицательным числам. Если есть отрицательные, добавление элемента способно уменьшить сумму, и рассуждение про сдвиг границы ломается. Тогда нужны префиксные суммы.


### Примеры подслучаев

Самое длинное окно: для [1, 2, 1, 1, 3] и K = 4 ответ 4, это отрезок [1, 2, 1, 1]. Когда добавляется 3, сумма становится 8 и левую границу надо сдвигать несколько раз.

Самое короткое окно: для [2, 3, 1, 2, 4, 3] и S = 7 ответ 2, это [4, 3]. Как только сумма стала не меньше S, сжимайте окно, пока оно остаётся достаточным.

Строка без повторов: в «pwwkew» окно сначала доходит до «pw», вторая w создаёт повтор, поэтому left сдвигается. Лучшие окна «wke» и «kew» имеют длину 3.


In [ ]:
a, limit = [1, 2, 1, 1, 3], 4
left = total = best = 0
for right, x in enumerate(a):
    total += x
    while total > limit:
        total -= a[left]
        left += 1
    best = max(best, right - left + 1)
print(best)

# Часть III. Практика

## Базовый уровень

### Задача 1. Максимум за один проход

**Идея перед задачей.** Один проход и текущий максимум.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 3 1 9 2 7
    --
    <- 9

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Один проход и текущий максимум.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task1

def task1():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task1(task1)

## Базовый уровень

### Задача 2. Индекс первого максимума

**Идея перед задачей.** Максимум и его первая позиция.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 3 9 1 9 2
    --
    <- 1

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Максимум и его первая позиция.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [1]:
from tests_algorithms import test_task2

def task2():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task2(task2)

Задача 2. Индекс первого максимума — тестов: 3
  ❌ тест 1: неверный ответ
     ввод: 5 | 3 9 1 9 2
     ожидалось: 1
     получено: (пусто)
  ❌ тест 2: неверный ответ
     ввод: 3 | -1 -2 -3
     ожидалось: 0
     получено: (пусто)
  ❌ тест 3: неверный ответ
     ввод: 1 | 42
     ожидалось: 0
     получено: (пусто)
Итог: пройдено 0 из 3 ❌


## Базовый уровень

### Задача 3. Соседние подъёмы

**Идея перед задачей.** Сравнение соседних элементов.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 1 3 2 2 5
    --
    <- 2

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Сравнение соседних элементов.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task3

def task3():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task3(task3)

## Базовый уровень

### Задача 4. Самая длинная серия

**Идея перед задачей.** Текущая серия и рекорд.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 8
    -> 1 1 2 2 2 3 3 1
    --
    <- 3

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Текущая серия и рекорд.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task4

def task4():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task4(task4)

## Базовый уровень

### Задача 5. Префиксные суммы

**Идея перед задачей.** Накопленная сумма.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 2 -1 3 0 4
    --
    <- 2 1 4 4 8

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Накопленная сумма.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [2]:
from tests_algorithms import test_task5

def task5():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task5(task5)

Задача 5. Префиксные суммы — тестов: 2
  ❌ тест 1: неверный ответ
     ввод: 5 | 2 -1 3 0 4
     ожидалось: 2 1 4 4 8
     получено: (пусто)
  ❌ тест 2: неверный ответ
     ввод: 3 | -1 -2 -3
     ожидалось: -1 -3 -6
     получено: (пусто)
Итог: пройдено 0 из 2 ❌


## Базовый уровень

### Задача 6. Сумма на отрезке

**Идея перед задачей.** Префикс с начальным нулём.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 2 -1 3 0 4
    -> 1 3
    --
    <- 2

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Префикс с начальным нулём.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task6

def task6():
    n=int(input())
    a=list(map(int,input().split()))
    l,r=map(int,input().split())
    ...

test_task6(task6)

## Базовый уровень

### Задача 7. Равновесный индекс

**Идея перед задачей.** Общая сумма и левая часть.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 1 3 5 2 2
    --
    <- 2

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Общая сумма и левая часть.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task7

def task7():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task7(task7)

## Базовый уровень

### Задача 8. Слияние двух списков

**Идея перед задачей.** Два указателя в двух отсортированных списках.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 3
    -> 1 4 8
    -> 4
    -> 2 3 7 9
    --
    <- 1 2 3 4 7 8 9

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Два указателя в двух отсортированных списках.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task8

def task8():
    n=int(input())
    a=list(map(int,input().split()))
    m=int(input())
    b=list(map(int,input().split()))
    ...

test_task8(task8)

## Базовый уровень

### Задача 9. Пара с заданной суммой

**Идея перед задачей.** Два указателя на краях отсортированного массива.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 1 2 4 7 11
    -> 9
    --
    <- YES

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Два указателя на краях отсортированного массива.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [3]:
from tests_algorithms import test_task9

def task9():
    n=int(input())
    a=list(map(int,input().split()))
    x=int(input())
    ...

test_task9(task9)

Задача 9. Пара с заданной суммой — тестов: 3
  ❌ тест 1: неверный ответ
     ввод: 5 | 1 2 4 7 11 | 9
     ожидалось: YES
     получено: (пусто)
  ❌ тест 2: неверный ответ
     ввод: 4 | 1 3 5 8 | 20
     ожидалось: NO
     получено: (пусто)
  ❌ тест 3: неверный ответ
     ввод: 4 | -3 -1 0 4 | 1
     ожидалось: YES
     получено: (пусто)
Итог: пройдено 0 из 3 ❌


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/mkir/.pyenv/versions/3.13.14/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 566, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/mkir/.pyenv/versions/3.13.14/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/Users/mkir/.pyenv/versions/3.13.14/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/mkir/.pyenv/versions/3.13.14/lib/python3.13/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/mkir/.pyenv/versions/3

## Базовый уровень

### Задача 10. Удалить дубликаты

**Идея перед задачей.** Сравнение с предыдущим числом.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 7
    -> 1 1 2 2 2 4 5
    --
    <- 1 2 4 5

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Сравнение с предыдущим числом.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task10

def task10():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task10(task10)

## Базовый уровень

### Задача 11. Палиндром без пробелов

**Идея перед задачей.** Нормализация строки и два указателя.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> А роза упала на лапу Азора
    --
    <- YES

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Нормализация строки и два указателя.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task11

def task11():
    s=input()
    ...

test_task11(task11)

## Базовый уровень

### Задача 12. Самый длинный отрезок

**Идея перед задачей.** Скользящее окно с неотрицательными числами.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 6
    -> 1 2 1 1 1 3
    -> 4
    --
    <- 4

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Скользящее окно с неотрицательными числами.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task12

def task12():
    n=int(input())
    a=list(map(int,input().split()))
    k=int(input())
    ...

test_task12(task12)

## Базовый уровень

### Задача 13. Подотрезки с суммой S

**Идея перед задачей.** Словарь частот префиксных сумм.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 1 2 1 2 1
    -> 3
    --
    <- 4

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Словарь частот префиксных сумм.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task13

def task13():
    n=int(input())
    a=list(map(int,input().split()))
    target=int(input())
    ...

test_task13(task13)

## Базовый уровень

### Задача 14. Ближайшая пара

**Идея перед задачей.** Двигайте указатель меньшего значения.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 3
    -> 1 4 10
    -> 3
    -> 2 15 20
    --
    <- 1 2

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Двигайте указатель меньшего значения.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task14

def task14():
    n=int(input())
    a=list(map(int,input().split()))
    m=int(input())
    b=list(map(int,input().split()))
    ...

test_task14(task14)

## Базовый уровень

### Задача 15. Нули в конец

**Идея перед задачей.** Соберите ненулевые и добавьте нули.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 7
    -> 0 1 0 3 12 0 5
    --
    <- 1 3 12 5 0 0 0

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Соберите ненулевые и добавьте нули.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task15

def task15():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task15(task15)

## Средний уровень

### Задача 16. Минимальная длина окна

**Идея перед задачей.** Сжимайте окно, когда сумма уже достаточна.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 6
    -> 2 3 1 2 4 3
    -> 7
    --
    <- 2

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Сжимайте окно, когда сумма уже достаточна.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task16

def task16():
    n=int(input())
    a=list(map(int,input().split()))
    target=int(input())
    ...

test_task16(task16)

## Средний уровень

### Задача 17. Подстрока без повторов

**Идея перед задачей.** Окно и частоты символов.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> abcabcbb
    --
    <- 3

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Окно и частоты символов.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task17

def task17():
    s=input()
    ...

test_task17(task17)

## Средний уровень

### Задача 18. Подотрезки с суммой не больше K

**Идея перед задачей.** После сжатия окна посчитайте все допустимые начала.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 1 2 1 1 3
    -> 4
    --
    <- 11

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>После сжатия окна посчитайте все допустимые начала.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task18

def task18():
    n=int(input())
    a=list(map(int,input().split()))
    k=int(input())
    ...

test_task18(task18)

## Средний уровень

### Задача 19. Серия после замен нулей

**Идея перед задачей.** Окно с количеством нулей не больше K.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 7
    -> 1 0 1 1 0 0 1
    -> 2
    --
    <- 6

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Окно с количеством нулей не больше K.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task19

def task19():
    n=int(input())
    a=list(map(int,input().split()))
    k=int(input())
    ...

test_task19(task19)

## Средний уровень

### Задача 20. Анаграммы

**Идея перед задачей.** Словарь частот букв.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> listen
    -> silent
    --
    <- YES

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Словарь частот букв.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task20

def task20():
    first=input()
    second=input()
    ...

test_task20(task20)

## Средний уровень

### Задача 21. Три числа с суммой

**Идея перед задачей.** Сортировка и два указателя.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 6
    -> -1 0 1 2 -1 -4
    -> 0
    --
    <- YES

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Сортировка и два указателя.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task21

def task21():
    n=int(input())
    a=list(map(int,input().split()))
    target=int(input())
    ...

test_task21(task21)

## Средний уровень

### Задача 22. Дождевая вода

**Идея перед задачей.** Максимумы слева и справа.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 12
    -> 0 1 0 2 1 0 1 3 2 1 2 1
    --
    <- 6

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Максимумы слева и справа.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task22

def task22():
    n=int(input())
    heights=list(map(int,input().split()))
    ...

test_task22(task22)

## Средний уровень

### Задача 23. Минимальное окно букв

**Идея перед задачей.** Частоты требуемых символов в окне.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> ADOBECODEBANC
    -> ABC
    --
    <- 4

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Частоты требуемых символов в окне.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task23

def task23():
    s=input()
    t=input()
    ...

test_task23(task23)

## Средний уровень

### Задача 24. Количество инверсий

**Идея перед задачей.** Сортировка слиянием.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 2 4 1 3 5
    --
    <- 3

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Сортировка слиянием.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task24

def task24():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task24(task24)

## Средний уровень

### Задача 25. Максимальная сумма подотрезка

**Идея перед задачей.** Алгоритм Кадане.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 9
    -> -2 1 -3 4 -1 2 1 -5 4
    --
    <- 6

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Алгоритм Кадане.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task25

def task25():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task25(task25)

## Сложный уровень

### Задача 26. Первый пропавший положительный

**Идея перед задачей.** Размещение числа x на позиции x - 1.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 4
    -> 1 2 0 4
    --
    <- 3

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Размещение числа x на позиции x - 1.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task26

def task26():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task26(task26)

## Сложный уровень

### Задача 27. Максимальное произведение

**Идея перед задачей.** Текущий максимум и минимум.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 4
    -> 2 3 -2 4
    --
    <- 6

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Текущий максимум и минимум.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task27

def task27():
    n=int(input())
    a=list(map(int,input().split()))
    ...

test_task27(task27)

## Сложный уровень

### Задача 28. Ровно K нечётных

**Идея перед задачей.** Префиксы, где нечётное число равно единице.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 1 1 2 1 1
    -> 3
    --
    <- 2

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Префиксы, где нечётное число равно единице.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task28

def task28():
    n=int(input())
    a=list(map(int,input().split()))
    k=int(input())
    ...

test_task28(task28)

## Сложный уровень

### Задача 29. Медиана двух списков

**Идея перед задачей.** Слияние отсортированных списков.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 2
    -> 1 3
    -> 1
    -> 2
    --
    <- 2

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Слияние отсортированных списков.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task29

def task29():
    n=int(input())
    a=list(map(int,input().split()))
    m=int(input())
    b=list(map(int,input().split()))
    ...

test_task29(task29)

## Сложный уровень

### Задача 30. Ровно K различных

**Идея перед задачей.** Не более K минус не более K - 1.

Напишите программу, которая решает задачу за время, подходящее для массива длины до 100000. Не добавляйте пояснений к печатаемому ответу.

**Вход.** Формат чтения уже показан в шаблоне ниже.

**Выход.** Напечатайте ответ ровно в требуемом формате.

**Пример.**

    -> 5
    -> 1 2 1 2 3
    -> 2
    --
    <- 7

<details><summary>Подсказка 1</summary>Назовите, что означает каждая переменная после очередного шага алгоритма.</details>

<details><summary>Подсказка 2</summary>Не более K минус не более K - 1.</details>

<details><summary>Разбор и эталонный код</summary>Решение строится на указанном приёме. Проверьте на одном элементе, одинаковых значениях и на случае без подходящего ответа. После того как каждый указатель проходит массив ограниченное число раз, сложность получается O(n), если не требуется сортировка.</details>

In [ ]:
from tests_algorithms import test_task30

def task30():
    n=int(input())
    a=list(map(int,input().split()))
    k=int(input())
    ...

test_task30(task30)